# GuitarSet player-held-out development: F and oracle E
Train players 00–03; validation 05; reserved player 04 is not loaded. All three progressions occur in training. Separate protocol from 02h, not a published benchmark or pristine independent test. Read docs/p2/guitarset/GUITARSET_PLAYER_EXPERIMENT.md. Enable Internet and GPU.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib
project=Path('/kaggle/working/chord-practice-system')
if not project.exists():
    subprocess.run(['git','clone','https://github.com/PhatLavar/chord-practice-system.git',str(project)],check=True)
os.chdir(project)
subprocess.run(['git','pull','--ff-only','origin','main'],check=True)
subprocess.run([sys.executable,"-m","pip","install","--no-deps","-e","."],check=True)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-audio.txt'],check=True)
import torch
print('torch',torch.__version__,'CUDA',torch.cuda.is_available())
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests'],check=True)


In [ ]:
# Official pinned archives; about 696 MB total. No extraction to separate WAV files.

from acr.data.guitarset.audit import download
from acr.data.guitarset.cqt import load_cache
raw=Path('data/raw/guitarset')
download(raw)
config='configs/p2/p2-guitarset-cqt-player.json'
cfg=json.loads(Path(config).read_text())
def prepare_cache(path, smoke=False):
    if not path.exists():
        command=[sys.executable,'-m', 'acr.data.guitarset.cqt','--output',str(path),'--config',config]
        if smoke: command.append('--smoke')
        subprocess.run(command,check=True)
    # Checks archive identity recorded in cache, hashes, folds and train-only stats.
    songs,refs,meta=load_cache(path,cfg,smoke)
    print({'cache':str(path),'recordings':len(songs),'fold':cfg['fold'],'smoke':smoke})
    del songs, refs


In [ ]:
# Short CPU execution checks, separate from the full experiment.
smoke_cache=Path('data/processed/guitarset-cqt-player-smoke')
prepare_cache(smoke_cache,smoke=True)
revision=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
source_hashes={p.as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in Path('src/acr').rglob('*.py')}
def run_stage(cache, output, smoke=False):
    if output.exists():
        assert (output/'summary.json').exists(), f'Incomplete run: {output}; preserve and inspect before choosing a fresh output.'
        for teacher in ('F','E'):
            record=json.loads((output/teacher/'run.json').read_text())
            assert record['config']==cfg and record['smoke']==smoke
            assert record['git_revision']==revision and record['source_sha256']==source_hashes
        print('Reusing completed run',output)
        return
    command=[sys.executable,'-m', 'acr.training.guitarset','--cache',str(cache),'--output',str(output),
             '--config',config,'--device','cpu' if smoke else 'cuda']
    if smoke: command.append('--smoke')
    subprocess.run(command,check=True)
run_stage(smoke_cache,Path('runs/p2-guitarset-cqt-player-kaggle-smoke'),smoke=True)


In [ ]:
# Prepare 240 train + 60 validation recordings. No outer-test features.
cache=Path('data/processed/guitarset-cqt-player')
prepare_cache(cache)


In [ ]:
assert torch.cuda.is_available(), 'Select a GPU for the full two-model pilot.'
run_root=Path('runs/p2-guitarset-cqt-player')
run_stage(cache,run_root)


In [ ]:
summary=json.loads((run_root/'summary.json').read_text())
for teacher,result in summary.items():
    print(teacher,'best_epoch',result['best_epoch'],
          {name:round(values['score'],4) if values['score'] is not None else None for name,values in result['metrics'].items()},
          'supported_oracle_event_accuracy',result['supported_oracle_event_accuracy'])
print('F: predicted frame timeline. E: gold event boundaries. Validation only; not published benchmark scores.')


In [ ]:
import shutil
archive=shutil.make_archive('/kaggle/working/p2-guitarset-cqt-player-results','zip',project,'runs/p2-guitarset-cqt-player')
print(archive)


Save the executed notebook version. Download **p2-guitarset-cqt-player-results.zip** and place it in local runs/imports/. It includes checkpoints, histories, provenance, normalization, validation references, logits, predicted LAB files and duration-weighted metrics. Raw audio and CQT cache are not bundled. A partial run is not automatically resumed; completed stages are reused only with matching revision/config/source hashes.
